# XManager with Vertex Training Cluster

This notebook demonstrates how to use XManager to launch and manage distributed training jobs on **Slurm-based Vertex Training Clusters**.

## Features Covered

1. **Experiment Management** - Create, track, and list experiments
2. **Three Submission Modes** - Auto-generate, Template, Raw Script
3. **Cluster Configurations** - NCCL settings for different GPU types
4. **Job Lifecycle** - Submit, monitor, cancel jobs
5. **Log Streaming** - Real-time output via SSH

## Supported Cluster Types

| Cluster Type | GPU | GPUs/Node | Network |
|-------------|-----|-----------|----------|
| `hcc-a3m` | H100 | 8 | TCPXO |
| `hcc-a3u` | H200 | 8 | gIB |
| `hcc-a4` | B200 | 8 | gIB |

---
## 1. Setup and Installation

In [ ]:
# Install dependencies
!pip install nest_asyncio pandas -q

# Install XManager (if not already installed)
# !pip install xmanager

# For development, install from local source
# !pip install -e /path/to/xmanager

In [1]:
# Core imports
from xmanager import xm
from xmanager import xm_local

# For displaying results
import pandas as pd
from IPython.display import display, HTML

# Initialize absl flags (required for XManager in notebooks)
import sys
from absl import flags

# Parse flags with empty argv to initialize XManager
# This is needed because XManager uses absl flags internally
if not flags.FLAGS.is_parsed():
    flags.FLAGS(sys.argv[:1])  # Only pass the script name, ignore notebook args

# Enable nested asyncio for Jupyter compatibility
# XManager uses asyncio internally, which conflicts with Jupyter's event loop
import nest_asyncio
nest_asyncio.apply()

print("XManager initialized successfully!")

XManager initialized successfully!


---
## 2. Configuration

Set your cluster connection details and job parameters.

In [2]:
# =============================================================================
# CLUSTER CONNECTION - Update these for your environment
# =============================================================================

# SSH connection to cluster login node
LOGIN_NODE = "vmdsa405-login-001"  # Cluster login node name
USE_GCLOUD_SSH = True  # Use gcloud compute ssh (vs direct ssh)
SSH_HOSTNAME = "nic0.vmdsa405-login-001.asia-southeast1-b.c.ai-infra-recipe-validation.internal.gcpnode.com"

# =============================================================================
# CLUSTER SETTINGS
# =============================================================================

CLUSTER_TYPE = "hcc-a4"  # Options: hcc-a3m, hcc-a3u, hcc-a4, hcc-a3h
PARTITION = "a4"         # Slurm partition
ACCOUNT = "aaie"         # Slurm account (optional)
NUM_NODES = 2            # Number of nodes
TIME_LIMIT = "0"         # Time limit (0 = unlimited)

# =============================================================================
# PATHS ON CLUSTER
# =============================================================================

WORK_DIR = "/home/abhishekbhgwt_google_com/vertexai-mds/nemo"
CONTAINER_IMAGE = f"{WORK_DIR}/nemo-demo.sqsh"
RECIPE = "pretrain/llama3p1_2b_pt.py"

print("Configuration loaded!")
print(f"  Cluster: {CLUSTER_TYPE}")
print(f"  Partition: {PARTITION}")
print(f"  Nodes: {NUM_NODES}")
print(f"  Work dir: {WORK_DIR}")

Configuration loaded!
  Cluster: hcc-a4
  Partition: a4
  Nodes: 2
  Work dir: /home/abhishekbhgwt_google_com/vertexai-mds/nemo


---
## 3. Cluster Configuration Details

Each cluster type has pre-configured NCCL settings optimized for its networking.

In [3]:
# Create a minimal executor to inspect cluster config
executor = xm_local.VertexTrainingCluster(
    cluster_type=CLUSTER_TYPE,
    partition=PARTITION,
)

# Get the cluster configuration
cluster_config = executor.get_cluster_config()

print(f"Cluster Configuration for {CLUSTER_TYPE}")
print("=" * 50)
print(f"GPU Type: {cluster_config.gpu_type}")
print(f"GPUs per Node: {cluster_config.gpus_per_node}")
print(f"Total GPUs (with {NUM_NODES} nodes): {NUM_NODES * cluster_config.gpus_per_node}")
print()
print("NCCL Environment Variables:")
for key, value in cluster_config.nccl_env_vars.items():
    print(f"  {key}={value}")

Cluster Configuration for hcc-a4
GPU Type: B200
GPUs per Node: 8
Total GPUs (with 2 nodes): 16

NCCL Environment Variables:
  NCCL_IB_TC=52
  NCCL_IB_FIFO_TC=84
  NCCL_NVLS_CHUNKSIZE=524288
  NCCL_SOCKET_IFNAME=enp0s19,enp192s20
  NCCL_NET_GDR_LEVEL=PIX
  NCCL_NET=gIB
  NCCL_IB_GID_INDEX=3
  NCCL_P2P_NET_CHUNKSIZE=131072
  NCCL_IB_QPS_PER_CONNECTION=4
  NCCL_P2P_PCI_CHUNKSIZE=131072
  NCCL_P2P_NVL_CHUNKSIZE=524288
  NCCL_TUNER_CONFIG_PATH=/usr/local/gib/configs/tuner_config_a4.txtpb
  NCCL_IB_ADAPTIVE_ROUTING=1
  NCCL_CROSS_NIC=0


---
## 4. Submission Mode 1: Auto-Generate with NemoRunLauncher

The easiest way to launch NeMo training jobs. XManager automatically generates the sbatch script using the `NemoRunLauncher`.

In [4]:
# Create a NemoRunLauncher - this handles sbatch script generation
nemo_launcher = xm_local.launchers.NemoRunLauncher(
    recipe=RECIPE,
    container_image=CONTAINER_IMAGE,
    experiment_name="llama3p1_pretrain_xmanager",
    extra_args=["--account", ACCOUNT] if ACCOUNT else [],
)

print("NemoRunLauncher configured:")
print(f"  Recipe: {nemo_launcher.recipe}")
print(f"  Container: {nemo_launcher.container_image}")
print(f"  Extra args: {nemo_launcher.extra_args}")

NemoRunLauncher configured:
  Recipe: pretrain/llama3p1_2b_pt.py
  Container: /home/abhishekbhgwt_google_com/vertexai-mds/nemo/nemo-demo.sqsh
  Extra args: ['--account', 'aaie']


In [5]:
# Launch job with NemoRunLauncher (auto-generate mode)
import time
timestamp = time.strftime("%Y%m%d-%H%M%S")
async with xm_local.create_experiment(experiment_title=f"vtc_nemo_xmanager_launch_{timestamp}") as experiment:
    
    # Create executor with launcher
    executor = xm_local.VertexTrainingCluster(
        cluster_type=CLUSTER_TYPE,
        partition=PARTITION,
        account=ACCOUNT,
        time_limit=TIME_LIMIT,
        
        # Use NemoRunLauncher for auto-generation
        launcher=nemo_launcher,
        
        # Resource requirements
        requirements=xm.JobRequirements(replicas=NUM_NODES),
        
        # Connection
        login_node=LOGIN_NODE,
        use_gcloud_ssh=USE_GCLOUD_SSH,
        ssh_hostname=SSH_HOSTNAME,
        
        # Paths
        work_dir=WORK_DIR,
        
        # Enable log streaming
        stream_output=True,
    )
    
    # Create and add job
    job = xm.Job(
        executable=xm.Binary(path=RECIPE),
        executor=executor,
    )
    
    await experiment.add(xm.JobGroup(job=job))
    
    experiment_id = experiment.experiment_id
    print(f"\nExperiment ID: {experiment_id}")
    print("Job submitted via auto-generated sbatch script!")

Submitted Slurm job 89 for exp1768350366796_1768350366796_1

Experiment ID: 1768350366796
Job submitted via auto-generated sbatch script!
Waiting for local jobs to complete. Press Ctrl+C to terminate them and exit


[exp1768350366796_1768350366796_1] tail: cannot open '/home/abhishekbhgwt_google_com/vertexai-mds/nemo/slurm-89.out' for reading: No such file or directory
[exp1768350366796_1768350366796_1] tail: no files remaining


---
## [DO NOT USE] 5. Submission Mode 2: Custom Jinja2 Template

For more control, provide a Jinja2 template. XManager fills in the variables.

In [ ]:
# Define a custom sbatch template
CUSTOM_TEMPLATE = '''#!/bin/bash
#SBATCH --job-name={{ job_name }}
#SBATCH --partition={{ partition }}
#SBATCH --nodes={{ num_nodes }}
#SBATCH --ntasks-per-node=1
#SBATCH --gpus-per-node={{ gpus_per_node }}
#SBATCH --time={{ time_limit }}
#SBATCH --exclusive
#SBATCH --output={{ log_dir }}/slurm-%j.out
{% if account %}#SBATCH --account={{ account }}{% endif %}

# Set up environment
export MASTER_ADDR=$(scontrol show hostname $SLURM_NODELIST | head -n1)
export MASTER_PORT=29500

# NCCL configuration
{% for key, value in nccl_env_vars.items() %}
export {{ key }}={{ value }}
{% endfor %}

echo "========================================"
echo "Job ID: $SLURM_JOB_ID"
echo "Nodes: $SLURM_NNODES"
echo "Master: $MASTER_ADDR:$MASTER_PORT"
echo "========================================"

# Run training
cd {{ work_dir }}
srun --container-image={{ container_image }} \
     --container-mounts={{ work_dir }}:{{ work_dir }} \
     python {{ script }} {{ script_args }}
'''

print("Custom template defined!")
print("Available variables: job_name, partition, num_nodes, gpus_per_node, ")
print("                     time_limit, account, log_dir, nccl_env_vars, ")
print("                     work_dir, container_image, script, script_args")

In [ ]:
# Launch job with custom template
async with xm_local.create_experiment(experiment_title="vtc_custom_template") as experiment:
    
    executor = xm_local.VertexTrainingCluster(
        cluster_type=CLUSTER_TYPE,
        partition=PARTITION,
        account=ACCOUNT,
        
        # Use custom template instead of launcher
        sbatch_template=CUSTOM_TEMPLATE,
        
        # Container
        container_image=CONTAINER_IMAGE,
        
        requirements=xm.JobRequirements(replicas=NUM_NODES),
        
        login_node=LOGIN_NODE,
        use_gcloud_ssh=USE_GCLOUD_SSH,
        ssh_hostname=SSH_HOSTNAME,
        work_dir=WORK_DIR,
    )
    
    job = xm.Job(
        executable=xm.Binary(path="train.py"),
        executor=executor,
        args={"batch_size": 32, "learning_rate": 1e-4},
    )
    
    await experiment.add(xm.JobGroup(job=job))
    print(f"Experiment ID: {experiment.experiment_id}")

---
##[DO NOT USE] 6. Submission Mode 3: Raw Script

For complete control, provide the entire sbatch script as a string.

In [ ]:
# Define a complete raw sbatch script
RAW_SCRIPT = f'''#!/bin/bash
#SBATCH --job-name=xmanager_raw
#SBATCH --partition={PARTITION}
#SBATCH --nodes={NUM_NODES}
#SBATCH --ntasks-per-node=1
#SBATCH --gpus-per-node=8
#SBATCH --time=01:00:00
#SBATCH --exclusive
#SBATCH --account={ACCOUNT}

echo "Hello from raw sbatch script!"
echo "Running on $SLURM_NNODES nodes"
echo "Job ID: $SLURM_JOB_ID"

# Your custom training command here
srun hostname
'''

print("Raw script defined!")
print(RAW_SCRIPT)

In [ ]:
# Launch job with raw script
async with xm_local.create_experiment(experiment_title="vtc_raw_script") as experiment:
    
    executor = xm_local.VertexTrainingCluster(
        cluster_type=CLUSTER_TYPE,
        
        # Use raw script - complete control
        sbatch_script=RAW_SCRIPT,
        
        login_node=LOGIN_NODE,
        use_gcloud_ssh=USE_GCLOUD_SSH,
        ssh_hostname=SSH_HOSTNAME,
        work_dir=WORK_DIR,
    )
    
    job = xm.Job(
        executable=xm.Binary(path="ignored_for_raw_script"),
        executor=executor,
    )
    
    await experiment.add(xm.JobGroup(job=job))
    print(f"Experiment ID: {experiment.experiment_id}")

---
## 7. Experiment Tracking

XManager tracks all experiments in a local database. You can list, query, and retrieve experiment details.

In [6]:
# List all experiments
experiments = xm_local.list_experiments()

# Convert to DataFrame for nice display
exp_data = []
for exp in experiments[-10:]:  # Last 10 experiments
    exp_data.append({
        "ID": exp.experiment_id,
        "Title": exp._experiment_title,
    })

df = pd.DataFrame(exp_data)
print("Recent Experiments:")
display(df)

Recent Experiments:


,ID,Title
0,1768346372367,vtc_nemo_autogen
1,1768346389467,vtc_nemo_autogen
2,1768346554481,vtc_nemo_autogen
3,1768348435715,vtc_nemo_xmanager_launch_20260113-235355
4,1768349450344,vtc_nemo_xmanager_launch_20260114-001050
5,1768349873351,vtc_nemo_xmanager_launch_20260114-001753
6,1768349969843,vtc_nemo_xmanager_launch_20260114-001929
7,1768350152935,vtc_nemo_xmanager_launch_20260114-002232
8,1768350244315,vtc_nemo_xmanager_launch_20260114-002404
9,1768350366796,vtc_nemo_xmanager_launch_20260114-002606


In [4]:
# Get a specific experiment by ID
# Replace with an actual experiment ID from your list
EXPERIMENT_ID = 1768350366796  # Use the one we just created

exp = xm_local.get_experiment(EXPERIMENT_ID)
print(f"Experiment: {exp._experiment_title}")
print(f"ID: {exp.experiment_id}")

work_units = exp._experiment_units                                                                                                                                                                                                                                                                                                                              
print(f"\nWork Units: {len(work_units)}")                                                                                                                                                                                                                                                                                                                       
for wu in work_units:                                                                                                                                                                                                                                                                                                                                           
    print(f"  - Work Unit {wu.work_unit_id}")                                                                                                                                                                                                                                                                                                                   
    # Show execution handles if available                                                                                                                                                                                                                                                                                                                       
    if hasattr(wu, '_non_local_execution_handles'):                                                                                                                                                                                                                                                                                                             
        for handle in wu._non_local_execution_handles:                                                                                                                                                                                                                                                                                                          
            if hasattr(handle, 'slurm_job_id'):                                                                                                                                                                                                                                                                                                                 
                print(f"    Slurm Job ID: {handle.slurm_job_id}")     

Experiment: vtc_nemo_xmanager_launch_20260114-002606
ID: 1768350366796

Work Units: 1
  - Work Unit 1
    Slurm Job ID: 89


---
## 8. Job Monitoring

Monitor job status using XManager's built-in tracking. Jobs are tracked via `squeue` and `sacct`.

In [5]:
# Check job status via XManager                                                                                                                                                                                                                                                                                                                                                                     
# This queries squeue/sacct on the cluster                                                                                                                                                                                                                                                                                                                                                          
                                                                                                                                                                                                                                                                                                                                                                                                    
exp = xm_local.get_experiment(EXPERIMENT_ID)                                                                                                                                                                                                                                                                                                                                                        
                                                                                                                                                                                                                                                                                                                                                                                                    
for wu in exp._experiment_units:                                                                                                                                                                                                                                                                                                                                                                    
    print(f"Work Unit {wu.work_unit_id}:")                                                                                                                                                                                                                                                                                                                                                          
                                                                                                                                                                                                                                                                                                                                                                                                    
    # Get execution handles                                                                                                                                                                                                                                                                                                                                                                         
    for handle in wu._non_local_execution_handles:                                                                                                                                                                                                                                                                                                                                                  
        if hasattr(handle, 'slurm_job_id'):                                                                                                                                                                                                                                                                                                                                                         
            print(f"  Slurm Job ID: {handle.slurm_job_id}")                                                                                                                                                                                                                                                                                                                                         
                                                                                                                                                                                                                                                                                                                                                                                                    
            # Debug: Show executor settings                                                                                                                                                                                                                                                                                                                                                         
            print(f"  Debug - login_node: {handle.executor.login_node}")                                                                                                                                                                                                                                                                                                                            
            print(f"  Debug - use_gcloud_ssh: {handle.executor.use_gcloud_ssh}")                                                                                                                                                                                                                                                                                                                    
            print(f"  Debug - ssh_hostname: {handle.executor.ssh_hostname}")                                                                                                                                                                                                                                                                                                                        
            print(f"  Debug - work_dir: {handle.executor.work_dir}")                                                                                                                                                                                                                                                                                                                                
                                                                                                                                                                                                                                                                                                                                                                                                    
            # Debug: Run squeue directly                                                                                                                                                                                                                                                                                                                                                            
            result = handle.client.run_command(f"squeue -j {handle.slurm_job_id} -h -o '%T'", set_nemorun_home=False)                                                                                                                                                                                                                                                                               
            print(f"  Debug - squeue returncode: {result.returncode}")                                                                                                                                                                                                                                                                                                                              
            print(f"  Debug - squeue stdout: '{result.stdout.strip()}'")                                                                                                                                                                                                                                                                                                                            
            print(f"  Debug - squeue stderr: '{result.stderr.strip()}'")                                                                                                                                                                                                                                                                                                                            
                                                                                                                                                                                                                                                                                                                                                                                                    
        if hasattr(handle, 'get_status'):                                                                                                                                                                                                                                                                                                                                                           
            try:                                                                                                                                                                                                                                                                                                                                                                                    
                status = handle.get_status()                                                                                                                                                                                                                                                                                                                                                        
                status_name = status._status.name if hasattr(status, '_status') else str(status)                                                                                                                                                                                                                                                                                                    
                message = status.message if hasattr(status, 'message') and status.message else ""                                                                                                                                                                                                                                                                                                   
                print(f"  Status: {status_name}")                                                                                                                                                                                                                                                                                                                                                   
                if message:                                                                                                                                                                                                                                                                                                                                                                         
                    print(f"  Message: {message}")                                                                                                                                                                                                                                                                                                                                                  
            except Exception as e:                                                                                                                                                                                                                                                                                                                                                                  
                print(f"  Status check failed: {e}")        

Work Unit 1:
  Slurm Job ID: 89
  Debug - login_node: vmdsa405-login-001
  Debug - use_gcloud_ssh: True
  Debug - ssh_hostname: nic0.vmdsa405-login-001.asia-southeast1-b.c.ai-infra-recipe-validation.internal.gcpnode.com
  Debug - work_dir: /home/abhishekbhgwt_google_com/vertexai-mds/nemo
  Debug - squeue returncode: 1
  Debug - squeue stdout: ''
  Debug - squeue stderr: 'slurm_load_jobs error: Invalid job id specified'
  Status: COMPLETED
  Message: Slurm job 89 completed successfully


In [7]:
# Alternatively, check status directly via SSH
import subprocess

def run_on_cluster(cmd):
    """Run a command on the cluster via SSH."""
    if USE_GCLOUD_SSH:
        ssh_cmd = [
            "gcloud", "compute", "ssh", LOGIN_NODE,
            "--", "-T",
            "-o", f"Hostname={SSH_HOSTNAME}",
            cmd
        ]
    else:
        ssh_cmd = ["ssh", LOGIN_NODE, cmd]
    
    result = subprocess.run(ssh_cmd, capture_output=True, text=True)
    return result.stdout, result.stderr

# Check running jobs
stdout, stderr = run_on_cluster("squeue --me")
print("Running Jobs (squeue --me):")
print(stdout if stdout else "No running jobs")

Running Jobs (squeue --me):
             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)



In [34]:
# Check job history
stdout, stderr = run_on_cluster("sacct -j 84 --format=JobID,JobName,State,ExitCode,Elapsed")
print("Recent Job History (sacct):")
print(stdout)

Recent Job History (sacct):
JobID           JobName      State ExitCode    Elapsed 
------------ ---------- ---------- -------- ---------- 
84           exp176834+    PENDING      0:0   00:00:00 



---
## 9. Log Access

Access job logs from the cluster.

In [8]:
# View logs for a specific Slurm job
SLURM_JOB_ID = "89"  # Replace with actual job ID

# Find log file
stdout, _ = run_on_cluster(f"ls -la {WORK_DIR}/slurm-{SLURM_JOB_ID}.out 2>/dev/null || echo 'Log file not found'")
print(stdout)

# View last 50 lines of log
stdout, _ = run_on_cluster(f"tail -50 {WORK_DIR}/slurm-{SLURM_JOB_ID}.out 2>/dev/null || echo 'Log file not found'")
print("\nLog tail:")
print(stdout)

-rw-rw-r-- 1 abhishekbhgwt_google_com abhishekbhgwt_google_com 2724 Jan 14 00:38 /home/abhishekbhgwt_google_com/vertexai-mds/nemo/slurm-89.out


Log tail:
Namespace(executor='slurm', image='/home/abhishekbhgwt_google_com/vertexai-mds/nemo/nemo-demo.sqsh', slurm_type='hcc-a4', partition='a4', account='aaie', nodes=2, nodelist=None, work_dir='/home/abhishekbhgwt_google_com/vertexai-mds/nemo', recipe_script='pretrain/llama3p1_2b_pt.py', recipe_args='', import_ckpt_script=None, export_ckpt_script=None, import_ckpt_args='', export_ckpt_args=None, data_dir=None, checkpoint_dir=None, log_dir='/home/abhishekbhgwt_google_com/vertexai-mds/nemo/logs', cache_dir='/home/abhishekbhgwt_google_com/vertexai-mds/nemo/cache', job_id=None, experiment_name='llama3p1_pretrain_xmanager')
─ Entering Experiment llama3p1_pretrain_xmanager with id: llama3p1_pretrain_x… ─
[00:38:38] Launching job llama3p1_pretrain_xmanager for        experiment.py:798
           experiment llama3p1_pretrain_xmanager              

---
## 10. Cancel Jobs

Cancel running jobs using XManager or directly via scancel.

In [47]:
# Cancel a specific job
SLURM_JOB_ID_TO_CANCEL = "85"  # Replace with job ID to cancel

# Uncomment to actually cancel:
# stdout, stderr = run_on_cluster(f"scancel {SLURM_JOB_ID_TO_CANCEL}")
# print(f"Cancelled job {SLURM_JOB_ID_TO_CANCEL}")

print(f"To cancel job {SLURM_JOB_ID_TO_CANCEL}, uncomment the lines above")

Cancelled job 85
To cancel job 85, uncomment the lines above


---
## 11. [DO NOT USE] Advanced: Custom Launcher

Create a custom launcher for non-NeMo workloads using `CustomLauncher`.

In [ ]:
# CustomLauncher allows arbitrary command templates
custom_launcher = xm_local.launchers.CustomLauncher(
    template="""
cd {work_dir} && \
torchrun \
    --nnodes={num_nodes} \
    --nproc_per_node={gpus_per_node} \
    --master_addr={master_addr} \
    --master_port={master_port} \
    {script} {args}
""",
    env_vars={
        "CUDA_VISIBLE_DEVICES": "0,1,2,3,4,5,6,7",
    }
)

print("CustomLauncher template:")
print(custom_launcher.template)

---
## 12. [DO NOT USE] Environment Variables and Mounts

Configure environment variables and container mounts.

In [ ]:
# Example with custom env vars and mounts
executor = xm_local.VertexTrainingCluster(
    cluster_type=CLUSTER_TYPE,
    partition=PARTITION,
    
    # Custom environment variables
    env_vars={
        "WANDB_API_KEY": "your-wandb-key",
        "HF_TOKEN": "your-huggingface-token",
        "CUDA_DEVICE_MAX_CONNECTIONS": "1",
    },
    
    # Container mounts
    container_mounts=[
        f"{WORK_DIR}:{WORK_DIR}",
        "/datasets:/datasets:ro",  # Read-only mount
    ],
    
    # Prologue commands (run before main script)
    prologue_commands=[
        "echo 'Starting training...'",
        "nvidia-smi",
    ],
    
    # Epilogue commands (run after main script)
    epilogue_commands=[
        "echo 'Training complete!'",
    ],
    
    requirements=xm.JobRequirements(replicas=NUM_NODES),
    login_node=LOGIN_NODE,
    use_gcloud_ssh=USE_GCLOUD_SSH,
    ssh_hostname=SSH_HOSTNAME,
    work_dir=WORK_DIR,
)

print("Executor configured with:")
print(f"  Env vars: {list(executor.env_vars.keys())}")
print(f"  Mounts: {executor.container_mounts}")
print(f"  Prologue: {len(executor.prologue_commands)} commands")
print(f"  Epilogue: {len(executor.epilogue_commands)} commands")

---
## Summary

This notebook demonstrated:

| Feature | Description |
|---------|-------------|
| **NemoRunLauncher** | Auto-generate sbatch scripts for NeMo training |
| **Custom Templates** | Jinja2 templates with variable substitution |
| **Raw Scripts** | Complete control with raw sbatch scripts |
| **CustomLauncher** | Arbitrary command templates for other frameworks |
| **Cluster Configs** | Pre-configured NCCL settings per cluster type |
| **Experiment Tracking** | Track experiments in local database |
| **Job Monitoring** | Query job status via squeue/sacct |
| **Log Streaming** | Real-time log access via SSH |

### Next Steps

- Explore the `xmanager list` CLI command for experiment management
- Check out the launcher.py example for command-line usage
- Customize templates for your specific training workflow